In [5]:
from flask import Flask, request, render_template_string
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

app = Flask(__name__)

HTML_PAGE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>Titanic Survival Predictor</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            background: linear-gradient(135deg, #0f172a, #1e293b);
            color: #fff;
            margin: 0;
            padding: 0;
        }
        .container {
            max-width: 900px;
            margin: 40px auto;
            background: rgba(255,255,255,0.08);
            padding: 30px;
            border-radius: 18px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.35);
            backdrop-filter: blur(10px);
        }
        h1 { text-align: center; margin-bottom: 10px; }
        p.desc { text-align: center; color: #cbd5e1; margin-bottom: 25px; }
        .grid {
            display: grid;
            grid-template-columns: repeat(2, 1fr);
            gap: 18px;
        }
        label {
            display: block;
            margin-bottom: 8px;
            font-weight: bold;
            color: #e2e8f0;
        }
        input, select {
            width: 100%;
            padding: 12px;
            border: none;
            border-radius: 10px;
            outline: none;
            font-size: 16px;
            box-sizing: border-box;
        }
        .full { grid-column: 1 / -1; }
        button {
            width: 100%;
            padding: 14px;
            border: none;
            border-radius: 10px;
            background: #22c55e;
            color: white;
            font-size: 18px;
            font-weight: bold;
            cursor: pointer;
            margin-top: 10px;
        }
        button:hover { background: #16a34a; }
        .result {
            margin-top: 25px;
            padding: 18px;
            border-radius: 12px;
            background: rgba(255,255,255,0.12);
        }
        .success { color: #4ade80; font-size: 22px; font-weight: bold; }
        .danger { color: #f87171; font-size: 22px; font-weight: bold; }
        .small { color: #cbd5e1; margin-top: 8px; }
        table {
            width: 100%;
            border-collapse: collapse;
            margin-top: 18px;
            background: rgba(255,255,255,0.06);
        }
        th, td {
            border: 1px solid rgba(255,255,255,0.15);
            padding: 10px;
            text-align: left;
        }
        th { background: rgba(255,255,255,0.12); }
        .note {
            margin-top: 18px;
            color: #cbd5e1;
            font-size: 14px;
        }
        @media (max-width: 700px) {
            .grid { grid-template-columns: 1fr; }
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>Titanic Survival Predictor</h1>
        <p class="desc">Enter passenger details below to predict whether the passenger survived.</p>

        <form method="POST" action="/predict" id="titanicForm">
            <div class="grid">
                <div>
                    <label for="pclass">Passenger Class</label>
                    <select name="pclass" id="pclass" required>
                        <option value="1">1</option>
                        <option value="2">2</option>
                        <option value="3" selected>3</option>
                    </select>
                </div>

                <div>
                    <label for="sex">Sex</label>
                    <select name="sex" id="sex" required>
                        <option value="male" selected>Male</option>
                        <option value="female">Female</option>
                    </select>
                </div>

                <div>
                    <label for="age">Age</label>
                    <input type="number" name="age" id="age" min="0" max="100" step="1" value="25" required />
                </div>

                <div>
                    <label for="fare">Fare</label>
                    <input type="number" name="fare" id="fare" min="0" max="600" step="0.01" value="32.00" required />
                </div>

                <div class="full">
                    <button type="submit">Predict Survival</button>
                </div>
            </div>
        </form>

        {% if result %}
        <div class="result">
            <div class="{{ 'success' if result.prediction == 1 else 'danger' }}">
                Predicted: {{ 'Survived' if result.prediction == 1 else 'Did Not Survive' }}
            </div>
            <div class="small">Survival probability: {{ result.probability }}</div>
            <div class="small">Model accuracy: {{ result.accuracy }}</div>

            <table>
                <tr><th>Field</th><th>Value</th></tr>
                <tr><td>Pclass</td><td>{{ result.input.Pclass }}</td></tr>
                <tr><td>Sex</td><td>{{ result.input.Sex }}</td></tr>
                <tr><td>Age</td><td>{{ result.input.Age }}</td></tr>
                <tr><td>Fare</td><td>{{ result.input.Fare }}</td></tr>
            </table>
        </div>
        {% endif %}

        <div class="note">
            Open this app at <b>http://127.0.0.1:5432</b>
        </div>
    </div>

    <script>
        document.getElementById("titanicForm").addEventListener("submit", function() {
            console.log("Form submitted");
        });
    </script>
</body>
</html>
"""

def train_model():
    df = pd.read_csv("Titanic-Dataset.csv")
    df = df.drop(columns=['PassengerId', 'SibSp', 'Embarked', 'Cabin', 'Name', 'Ticket'], errors='ignore')

    if 'Age' in df.columns:
        df['Age'] = df['Age'].fillna(df['Age'].median())
    if 'Fare' in df.columns:
        df['Fare'] = df['Fare'].fillna(df['Fare'].median())

    if 'Sex' in df.columns:
        le = LabelEncoder()
        df['Sex'] = le.fit_transform(df['Sex'])

    X = df.drop(columns=['Survived'])
    y = df['Survived']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    acc = accuracy_score(y_test, model.predict(X_test))
    return model, X.columns.tolist(), acc

model, feature_cols, accuracy = train_model()

@app.route("/", methods=["GET"])
def index():
    return render_template_string(HTML_PAGE)

@app.route("/predict", methods=["POST"])
def predict():
    pclass = int(request.form["pclass"])
    sex = request.form["sex"]
    age = float(request.form["age"])
    fare = float(request.form["fare"])

    sex_encoded = 1 if sex.lower() == "male" else 0

    user_data = {
        "Pclass": pclass,
        "Sex": sex_encoded,
        "Age": age,
        "Fare": fare
    }

    input_df = pd.DataFrame([user_data])

    for col in feature_cols:
        if col not in input_df.columns:
            input_df[col] = 0

    input_df = input_df[feature_cols]

    prediction = int(model.predict(input_df)[0])
    probability = float(model.predict_proba(input_df)[0][1])

    result = {
        "prediction": prediction,
        "probability": f"{probability:.2%}",
        "accuracy": f"{accuracy:.2%}",
        "input": {
            "Pclass": pclass,
            "Sex": sex,
            "Age": age,
            "Fare": fare
        }
    }

    return render_template_string(HTML_PAGE, result=result)

import threading

def run_app():
    app.run(host="127.0.0.1", port=5001, debug=False, use_reloader=False)

threading.Thread(target=run_app, daemon=True).start()

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5001 is in use by another program. Either identify and stop that program, or start the server with a different port.


data=pd.read_csv('tips')